# Extension: MC Dropout uncertainty for MT

MT (and the paper's models generally) output a single probability per factor per month with
no sense of how much to trust that number. Monte Carlo Dropout (Gal and Ghahramani, 2016) is
a way to get an uncertainty estimate essentially for free from a network that already uses
dropout for regularization.

**Normally**, dropout randomly zeroes a fraction of neurons *during training only*, so the
network can't lean on any single narrow path through its own representation — it's switched
off at prediction time so the network's full capacity is used.

**MC Dropout's reframe**: leave dropout switched *on* at prediction time, and instead of asking
the network once, ask it the same question 30-100 times. Each pass silences a different random
subset of neurons, so each pass can give a slightly different answer — not because the world
changed, but because each pass samples a slightly different internal path through what the
network learned. If the input sits in territory the model understands well, the passes agree
closely. If it doesn't, the passes scatter. That scatter is the signal: it's not "what's the
answer," it's "how much do internally-varied versions of my own reasoning agree with each
other" — an empirical, after-the-fact check on whether the model has a stable, well-supported
answer or is essentially guessing behind a confident-looking number.

**What this notebook does:**
1. Train an MT variant with dropout layers in the shared trunk — same walk-forward procedure
   as `Initial MT.ipynb`, nothing else changes.
2. At prediction time, run each month's predictors through the network 50 times with dropout
   forced on, per factor. The mean of those 50 passes is the point estimate (same role as MT's
   usual single output); the spread (std) is the new uncertainty signal.
3. Calibrate an uncertainty cutoff each fold from that fold's validation-period spread (e.g.
   flag the most-uncertain 20% of factor-months) — never from the test period, to avoid leakage.
4. Compare two ways of acting on the flag against the unweighted baseline: shrink position size
   in proportion to uncertainty (gentler), or sit out flagged months entirely (more aggressive).

Architecture note: the paper's MT uses batch normalization, not dropout — this is a deliberate
addition for this extension. Forcing `training=True` to activate dropout at prediction time
would *also* put batch norm into batch-statistics mode (a well-known MC-Dropout pitfall), so
this variant's shared trunk uses dropout in place of batch norm instead of combining both.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

import loading
import estimation as est

FACTOR_NAMES = est.FACTOR_NAMES

In [ ]:
# Same walk-forward estimation procedure as Initial MT.ipynb (paper Sec 3.2.4) — this notebook
# only changes what happens at *prediction* time, not the training/validation/test split.

data, feature_cols = est.build_labels_and_panel(
    loading.response_factors, loading.macro_predictors, loading.financial_predictors
)

print("data shape:", data.shape)
print("features:", len(feature_cols))
print("date range:", data.index.min().date(), "to", data.index.max().date())

In [ ]:
N_MC_SAMPLES = 50          # forward passes per prediction; paper's own NN ensembling uses 10 seeds, 30-100 is the usual MC Dropout range
UNCERTAINTY_QUANTILE = 0.80  # flag the most-uncertain 20% of factor-months, calibrated per fold on validation data


def build_mt_mcdropout_model(n_features, l1_value=0.01, learning_rate=0.001, dropout_rate=0.2, seed=0):
    """MT's shared trunk (4 hard-sharing layers, 32 units) and factor-specific heads (2 layers,
    8 units, per factor) are unchanged from build_mt_model in Initial MT.ipynb. The only
    architectural change: batch norm -> dropout in the shared trunk, so dropout can be forced
    on at prediction time (training=True) without also perturbing batch norm's statistics."""
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inputs = Input(shape=(n_features,), name='predictors')
    x = inputs
    for i in range(4):
        x = layers.Dense(32, kernel_regularizer=regularizers.l1(l1_value), name=f'shared_dense_{i+1}')(x)
        x = layers.ReLU(name=f'shared_relu_{i+1}')(x)
        x = layers.Dropout(dropout_rate, name=f'shared_dropout_{i+1}')(x)
    shared_latent = x

    outputs = []
    for factor in FACTOR_NAMES:
        f = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'{factor}_dense1')(shared_latent)
        f = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'{factor}_dense2')(f)
        f_out = layers.Dense(1, activation='sigmoid', name=f'{factor}_output')(f)
        outputs.append(f_out)

    model = Model(inputs=inputs, outputs=outputs, name='MT_MCDropout')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={f'{factor}_output': 'binary_crossentropy' for factor in FACTOR_NAMES},
    )
    return model


def mc_dropout_predict(model, X, n_samples=N_MC_SAMPLES):
    """Run n_samples stochastic forward passes with dropout forced on. Returns an array of
    shape (n_samples, n_rows, n_factors); take .mean(axis=0) for the point estimate and
    .std(axis=0) for the uncertainty."""
    X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
    samples = [
        np.concatenate([np.asarray(p) for p in model(X_tensor, training=True)], axis=1)
        for _ in range(n_samples)
    ]
    return np.stack(samples, axis=0)


# --- sanity check with fake data, same pattern as Initial MT.ipynb's build_mt_model check ---
n_features = len(feature_cols)
model = build_mt_mcdropout_model(n_features)
X_fake = np.random.randn(20, n_features).astype('float32')
mc_fake = mc_dropout_predict(model, X_fake, n_samples=10)
print("MC sample array shape (n_samples, n_rows, n_factors):", mc_fake.shape)
print("Model builds and MC-samples without error.")

In [ ]:
# Train: identical walk-forward loop to Initial MT.ipynb's train cell (expanding window,
# 2-year validation window before the test year, single seed, fixed hyperparameters — same
# tractability simplification vs. the paper's grid search + 10-seed ensemble noted there).
#
# The only addition: after training, run MC Dropout on the validation set to calibrate this
# fold's uncertainty cutoff (validation only, never test — avoids leaking test-period info into
# the flagging rule), then run it on the test set to get this fold's point estimates + uncertainty.

mean_records, std_records, flag_records = [], [], []

for test_year in est.OOS_TEST_YEARS:
    train, val, test, X_train, X_val, X_test = est.prepare_fold(data, feature_cols, test_year)

    y_train = {f'{f}_output': train[f'{f}_label'].values.astype('float32') for f in FACTOR_NAMES}
    y_val = {f'{f}_output': val[f'{f}_label'].values.astype('float32') for f in FACTOR_NAMES}

    model = build_mt_mcdropout_model(n_features=X_train.shape[1])
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(
        X_train.values.astype('float32'), y_train,
        validation_data=(X_val.values.astype('float32'), y_val),
        epochs=200, batch_size=4, callbacks=[es], verbose=0,
    )

    val_mc = mc_dropout_predict(model, X_val.values)
    val_std = val_mc.std(axis=0)  # (n_val, n_factors)
    cutoff = np.quantile(val_std.flatten(), UNCERTAINTY_QUANTILE)

    test_mc = mc_dropout_predict(model, X_test.values)
    test_mean, test_std = test_mc.mean(axis=0), test_mc.std(axis=0)
    test_flag = test_std > cutoff

    cols = [f'{f}_prob' for f in FACTOR_NAMES], [f'{f}_std' for f in FACTOR_NAMES], [f'{f}_uncertain' for f in FACTOR_NAMES]
    mean_records.append(pd.DataFrame(test_mean, index=test.index, columns=cols[0]))
    std_records.append(pd.DataFrame(test_std, index=test.index, columns=cols[1]))
    flag_records.append(pd.DataFrame(test_flag, index=test.index, columns=cols[2]))

    print(f'{test_year}: cutoff={cutoff:.4f}  mean test uncertainty={test_std.mean():.4f}  '
          f'flagged={test_flag.mean():.0%}')

oos_mean = pd.concat(mean_records).sort_index()
oos_std = pd.concat(std_records).sort_index()
oos_flag = pd.concat(flag_records).sort_index()

oos_mean.to_csv('../results/mcdropout_oos_predictions.csv')
oos_std.to_csv('../results/mcdropout_oos_uncertainty.csv')
print("\nOOS predictions:", oos_mean.shape)

In [ ]:
# Build three trading strategies from the same MC-Dropout predictions, to see whether the
# uncertainty signal is worth acting on:
#   baseline  - MT's usual rule: long if predicted prob > 0.5, flat otherwise (paper Eq. 3)
#   scaled    - baseline, but position size shrinks with uncertainty (gentler: smaller bet, not zero)
#   abstain   - baseline, but flagged (top 20% most uncertain) months are skipped entirely

response = loading.response_factors.rename(columns={'Mom': 'MOM'})
r = response.loc[oos_mean.index, FACTOR_NAMES]
signal = pd.DataFrame({f: (oos_mean[f'{f}_prob'] > 0.5).astype(int) for f in FACTOR_NAMES}, index=oos_mean.index)

baseline_strat = signal * r
baseline_strat['EW'] = baseline_strat[FACTOR_NAMES].mean(axis=1)

# per-factor min-max normalize uncertainty to a [0, 1] confidence weight (1 = certain, 0 = most
# uncertain observed for that factor). A factor with zero variation in a given fold's test window
# can't be discriminated between certain/uncertain months, so it defaults to full confidence.
std_vals = oos_std.rename(columns=lambda c: c.replace('_std', ''))
value_range = (std_vals.max() - std_vals.min()).replace(0, 1)
norm_std = (std_vals - std_vals.min()) / value_range
confidence_weight = (1 - norm_std).clip(0, 1)

scaled_strat = signal * confidence_weight * r
scaled_strat['EW'] = scaled_strat[FACTOR_NAMES].mean(axis=1)

flag_vals = oos_flag.rename(columns=lambda c: c.replace('_uncertain', '')).astype(bool)
abstain_strat = signal * (~flag_vals).astype(int) * r
abstain_strat['EW'] = abstain_strat[FACTOR_NAMES].mean(axis=1)

print("mean confidence weight on flagged vs. unflagged factor-months:")
print(f"  flagged:   {confidence_weight.values[flag_vals.values].mean():.2f}")
print(f"  unflagged: {confidence_weight.values[~flag_vals.values].mean():.2f}")

In [ ]:
# Benchmark: does acting on MC-Dropout uncertainty improve risk-adjusted performance relative
# to the unweighted baseline, and is the uncertainty signal actually informative (do flagged
# months really predict worse, i.e. lower accuracy)?

buy_ew = response.loc[oos_mean.index, FACTOR_NAMES].mean(axis=1)

perf = pd.DataFrame({
    'baseline (no uncertainty)': [est.annualized_sharpe(baseline_strat['EW']), *est.spanning_regression(baseline_strat['EW'], buy_ew)],
    'confidence-scaled': [est.annualized_sharpe(scaled_strat['EW']), *est.spanning_regression(scaled_strat['EW'], buy_ew)],
    'abstain on flagged': [est.annualized_sharpe(abstain_strat['EW']), *est.spanning_regression(abstain_strat['EW'], buy_ew)],
}, index=['Sharpe Ratio', 'alpha (annualized %)', 't(alpha)', 'beta', 'R2 (%)'])
print("Multi-factor timing performance vs. multi-factor BUY:")
print(perf.round(3))
print(f"\nmulti-factor BUY Sharpe: {est.annualized_sharpe(buy_ew):.2f}")

# calibration check: if MC-Dropout uncertainty is meaningful, flagged (high-uncertainty)
# factor-months should have *lower* classification accuracy than unflagged ones.
y_true = pd.DataFrame({f: data.loc[oos_mean.index, f'{f}_label'] for f in FACTOR_NAMES})
y_pred = signal
correct = (y_true == y_pred)

acc_flagged = correct.values[flag_vals.values].mean()
acc_unflagged = correct.values[~flag_vals.values].mean()
print(f"\nAccuracy on flagged (uncertain) factor-months:   {acc_flagged:.1%}  (n={flag_vals.values.sum()})")
print(f"Accuracy on unflagged (confident) factor-months: {acc_unflagged:.1%}  (n={(~flag_vals.values).sum()})")